### Setup für alle Aufgaben

In [1]:
from pymongo import MongoClient
from datetime import datetime
from bson import ObjectId

# Verbindung zur lokalen MongoDB
client = MongoClient("mongodb://localhost:27017/")

# Datenbank "university" auswählen
db = client["universitaet"]

# Beispiel-Collections
students = db["studenten"]
kurse = db["kurse"]
orte = db["orte"]

# Vor jeder Aufgabe ggf. alte Daten löschen (optional)
students.delete_many({})
kurse.delete_many({})
orte.delete_many({})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

#### Aufgabe 1: einfacher Insert

🗒️ Erklärung:<br>
insert_one() fügt genau ein Dokument in die Collection ein. MongoDB erzeugt automatisch ein _id-Feld vom Typ ObjectId.

In [2]:
# Ein einfaches Dokument einfügen
student = {"name": "Anna", "alter": 22, "eingeschrieben": True}
students.insert_one(student)

print("Student eingefügt:", student)

Student eingefügt: {'name': 'Anna', 'alter': 22, 'eingeschrieben': True, '_id': ObjectId('69c0f4c7b0c0fe903eb1e99b')}


#### Aufgabe 2: Mehrere Dokumente einfügen

🗒️ Erklärung:<br>
<ul>
    <li>insert_many() erlaubt das Einfügen mehrerer Dokumente auf einmal.</li>
    <li>Arrays (Listen) sind ein nativer Datentyp in MongoDB.</li>
</ul>

In [3]:
# Liste von Dokumenten
many_students = [
    {"name": "Jonas", "skills": ["Python", "MongoDB"], "alter": 24},
    {"name": "Lea", "skills": ["C++", "ML"], "alter": 26}
]

students.insert_many(many_students)

print("Mehrere Studierende eingefügt.")

Mehrere Studierende eingefügt.


#### Aufgabe 3: Dokumente abfraltern

🗒️ Erklärung:<br>
<ul>
    <li>$gt bedeutet greater than.</li>
    <li>Das zweite Argument ({"_id": 0, "name": 1, "alter": 1}) ist eine Projektion – zeigt nur bestimmte Felder.</li>
</ul>

In [4]:
# Studierende mit Alter > 23 suchen
result = students.find({"alter": {"$gt": 23}}, {"_id": 0, "name": 1, "alter": 1})

for doc in result:
    print(doc)


{'name': 'Jonas', 'alter': 24}
{'name': 'Lea', 'alter': 26}


#### Aufgabe 4: Dokumente ändern

🗒️ Erklärung:<br>
<ul>
    <li>update_many() ändert alle Dokumente, die die Bedingung erfüllen.</li>
    <li>Der Operator $set setzt oder erstellt das Feld eingeschrieben.</li>
</ul>

In [5]:
# Setze eingeschrieben = False für Studierende über 25 Jahre
students.update_many({"alter": {"$gt": 25}}, {"$set": {"eingeschrieben": False}})

print("Update ausgeführt.")

Update ausgeführt.


#### Aufgabe 5: Löschen eines Dokuments

🗒️ Erklärung: delete_one() löscht nur das erste passende Dokument.

In [6]:
students.delete_one({"name": "Jonas"})
print("Jonas wurde gelöscht.")

Jonas wurde gelöscht.


#### Aufgabe 6: Datumsfelder

Erklärung:<br>
<ul>
    <li>MongoDB speichert datetime-Objekte nativ als BSON-DateTime.</li>
    <li>datetime.utcnow() sorgt für UTC-Zeit.</li>
</ul>

In [7]:
students.update_one(
    {"name": "Anna"},
    {"$set": {"eingeschrieben_am": datetime.utcnow()}}
)

anna = students.find_one({"name": "Anna"})
print("📅 Annas Einschreibedatum:", anna["eingeschrieben_am"])

📅 Annas Einschreibedatum: 2026-03-23 08:07:56.398000


#### Aufgabe 7: Sortierung und Projektion

🗒️ Erklärung:<br>
<ul>
    <li>sort("alter", -1) sortiert absteigend.</li>
    <li>1 wäre aufsteigend.</li>
</ul>

In [8]:
for doc in students.find({}, {"_id": 0, "name": 1, "alter": 1}).sort("alter", -1):
    print(doc)

{'name': 'Lea', 'alter': 26}
{'name': 'Anna', 'alter': 22}


#### Aufgabe 8: Komplexes Dokument mit eingebetteten Objekten

🗒️ Erklärung: Eingebettete Objekte und Arrays sind typisch für MongoDBs „document model“.

In [9]:
kurs_doc = {
    "titel": "MongoDB Basics",
    "dozent": {"name": "Dr. Müller", "email": "mueller@example.com"},
    "studenten": [
        {"name": "Anna", "note": 1.7},
        {"name": "Lea", "note": 2.3}
    ]
}

kurse.insert_one(kurs_doc)
print("Kurs mit eingebetteten Objekten eingefügt.")

Kurs mit eingebetteten Objekten eingefügt.


#### Aufgabe 9: Suche in Arrays


🗒️ Erklärung: MongoDB durchsucht Arrays automatisch – wenn irgendein Eintrag die Bedingung erfüllt, wird das Dokument gefunden.

In [10]:
result = kurse.find({"studenten.note": {"$lt": 2.0}})
for doc in result:
    print("📘 Kurs:", doc["titel"])

📘 Kurs: MongoDB Basics


#### Aufgabe 10: Zählen und Gruppieren


🗒️ Erklärung:<br>
<ul>
    <li>$group gruppiert nach einem Feld.</li>
    <li>$sum: 1 zählt, wie viele Dokumente pro Gruppe existieren.</li>
</ul>

In [19]:
pipeline = [
    {"$group": {"_id": "$eingeschrieben", "count": {"$sum": 1}}}
]

for group in students.aggregate(pipeline):
    print(group)

{'_id': True, 'count': 1}
{'_id': False, 'count': 1}


#### Aufgabe 11: Arbeiten mit ObjectId


🗒️ Erklärung:<br>
<ul>
    <li>ObjectId ist der Standardtyp für _id.</li>
    <li>Zum manuellen Zugriff müssen Sie ihn mit bson.ObjectId() umwandeln.</li>
</ul>

In [ ]:
# Beispiel: ein Dokument holen und dann per _id erneut suchen
doc = students.find_one()
id_str = str(doc["_id"])

found = students.find_one({"_id": ObjectId(id_str)})
print("🎯 Gefunden per ObjectId:", found)

{'_id': ObjectId('69c0f721b0c0fe903eb1e9a2'), 'name': 'Anna'}


#### Aufgabe 12: Null-Werte & Existenzprüfung


🗒️ Erklärung:<br>
<ul>
    <li>$exists prüft, ob ein Feld existiert.</li>
    <li>None (Python) entspricht null (MongoDB).</li>
</ul>

In [ ]:
# Ein Dokument mit Nullwert hinzufügen
#students.insert_one({"name": "Tom", "phone": None})

# Suche: entweder null oder Feld fehlt
query = {"$or": [{"phone": None}, {"phone": {"$exists": False}}]}
for doc in students.find(query):
    print("☎️", doc)

☎️ {'_id': ObjectId('69c0f4c7b0c0fe903eb1e99b'), 'name': 'Anna', 'alter': 22, 'eingeschrieben': True, 'eingeschrieben_am': datetime.datetime(2026, 3, 23, 8, 7, 56, 398000)}
☎️ {'_id': ObjectId('69c0f4c9b0c0fe903eb1e99d'), 'name': 'Lea', 'skills': ['C++', 'ML'], 'alter': 26, 'eingeschrieben': False}
☎️ {'_id': ObjectId('69c0f5f6b0c0fe903eb1e99f'), 'name': 'Tom', 'phone': None}
☎️ {'_id': ObjectId('69c0f5f7b0c0fe903eb1e9a0'), 'name': 'Tom', 'phone': None}
☎️ {'_id': ObjectId('69c0f5f8b0c0fe903eb1e9a1'), 'name': 'Tom', 'phone': None}


#### Aufgabe 13: Lookup


🗒️ Erklärung:<br>
<ul>
    <li>$lookup verbindet zwei Collections ähnlich wie ein SQL-Join.</li>
    <li>Das Ergebnis enthält ein Array student_kurss.</li>
</ul>

In [35]:
# Beispiel-Daten
students.delete_many({})
noten = db["noten"]
noten.delete_many({})

s1 = students.insert_one({"name": "Anna"}).inserted_id
s2 = students.insert_one({"name": "Lea"}).inserted_id

noten.insert_many([
    {"student_id": s1, "kurs": "MongoDB"},
    {"student_id": s1, "kurs": "Python"},
    {"student_id": s2, "kurs": "C++"}
])

# Aggregation mit $lookup
pipeline = [
    {
        "$lookup": {
            "from": "noten",
            "localField": "_id",
            "foreignField": "student_id",
            "as": "student_kurse"
        }
    }
]

for result in students.aggregate(pipeline):
    print(result)


{'_id': ObjectId('69c0f721b0c0fe903eb1e9a2'), 'name': 'Anna', 'student_kurse': [{'_id': ObjectId('69c0f721b0c0fe903eb1e9a4'), 'student_id': ObjectId('69c0f721b0c0fe903eb1e9a2'), 'kurs': 'MongoDB'}, {'_id': ObjectId('69c0f721b0c0fe903eb1e9a5'), 'student_id': ObjectId('69c0f721b0c0fe903eb1e9a2'), 'kurs': 'Python'}]}
{'_id': ObjectId('69c0f721b0c0fe903eb1e9a3'), 'name': 'Lea', 'student_kurse': [{'_id': ObjectId('69c0f721b0c0fe903eb1e9a6'), 'student_id': ObjectId('69c0f721b0c0fe903eb1e9a3'), 'kurs': 'C++'}]}


#### Expertenaufgabe 14: GeoJSON und Geospatial Query


🗒️ Erklärung:<br>
<ul>
    <li>GeoJSON-Objekte brauchen typ: "Point" und koordinaten: [longitude, latitude].</li>
    <li>Mit $near finden Sie Punkte im Umkreis.</li>
</ul>

In [36]:
#evtl. import pymongo
from pymongo.errors import DuplicateKeyError

# Beispiel-Orte einfügen
try:
    orte.insert_one({
        "_id": 1,
        "name": "Campus Berlin",
        "location": {
            "type": "Point",
            "coordinates": [13.4050, 52.5200]
        }
    })
# evtl. except pymongo.errors.DuplicateKeyError:
except DuplicateKeyError:
    # überspringen
    pass
except Exception as e:
    print(e)



# 2dsphere-Index anlegen
orte.create_index([("location", "2dsphere")])

# Query: Orte in der Nähe von Koordinaten
query = {
    "location": {
        "$near": {
            "$geometry": {"type": "Point", "coordinates": [13.40, 52.52]},
            "$maxDistance": 2000  # Meter
        }
    }
}


for loc in orte.find(query):
    print("Orte in der Nähe:", loc)


Orte in der Nähe: {'_id': 1, 'name': 'Campus Berlin', 'location': {'type': 'Point', 'coordinates': [13.405, 52.52]}}
